# Optimized Product Quantization (OPQ) and Score-Aware Quantization (ScaNN) — companion notebook

The **narrative** half of the Python pillar. The tested implementation lives next door in
[`optimized_product_quantization.py`](optimized_product_quantization.py); here we import it and walk
the topic section by section, so every claim renders as executed output.

OPQ **is** product quantization with a learned rotation, so the PQ core — the per-subspace Lloyd
quantizer, the synthetic finance cloud, the variance-imbalanced cloud, and the raw/PCA/balanced
baselines — is imported from the previous topic and never reimplemented. The module owns the
numbers; its `test_*` assertions encode the theorems and `viz_constants()` prints what the
laboratory mirrors to the decimal.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "optimized-product-quantization",
              pathlib.Path("notebooks/optimized-product-quantization")):
    if (_cand / "optimized_product_quantization.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import optimized_product_quantization as opq

## Act I — the rotation

### 1. A rotation is a free isometry

An orthogonal $R$ preserves every pairwise distance and inner product, so rotating the database
**and** the queries leaves the nearest-neighbor / MIPS problem unchanged. Product quantization is
the $R = I$ special case, so the rotation can only *help*: $E_{\text{OPQ}} \le E_{\text{PQ}}$ always.

In [ ]:
opq.test_orthogonal_preserves_distances()
opq.test_rotation_isometry_keeps_recall()

### 2. Parametric OPQ balances the *product* of variances

Under a Gaussian/high-rate bound, the optimal rotation decorrelates (PCA) **and** equalizes the
*product* of per-subspace variances (the determinant) — not the *sum* the prerequisite's heuristic
balances. We measure the spread of per-subspace log-variances (zero = perfectly product-balanced).

In [ ]:
X = opq.variance_imbalanced_cloud(n=600, d=8, seed=1)
print(f"  raw contiguous blocks : log-var spread = {opq.subspace_logvar_spread(X, np.eye(8), 4):.3f}")
print(f"  parametric OPQ        : log-var spread = {opq.subspace_logvar_spread(X, opq.parametric_opq(X, 4), 4):.3f}")
opq.test_parametric_balances_determinant()

### 3. Non-parametric OPQ: alternating optimization

Without a Gaussian assumption, OPQ alternates two globally-solved subproblems: (i) fix $R$, train PQ
on the rotated data; (ii) fix the codebooks, update $R$ by the closed-form **Orthogonal Procrustes**
solution $R = V U^{\mathsf T}$ from $\mathrm{SVD}(X^{\mathsf T} Q)$. Each step is its subproblem's
global optimum, so the distortion is monotone non-increasing — to a *local* optimum, cross-checked
against `scipy.linalg.orthogonal_procrustes`.

In [ ]:
opq.test_procrustes_minimizes_frobenius()
opq.validate_against_procrustes()
opq.test_opq_trajectory_monotone()

### 4. The on-ramp, closed: raw / PCA-only / balanced heuristic / learned OPQ

On the prerequisite's exact variance-imbalanced cloud, we re-derive its raw/PCA/balanced baselines
(never hardcoded — provably one cloud) and run the learned OPQ rotation. The heuristic's $\sim 15.4$
becomes the *starting point* the alternating optimization descends below.

In [ ]:
base = opq.rotation_distortion_study(X, 4, 16, seed=1)
_, _, traj = opq.nonparametric_opq(X, 4, 16, n_iter=15, seed=1)
for name in ("raw", "pca_only", "balanced"):
    print(f"  PQ distortion ({name:9s}) = {base[name]:.4f}")
print(f"  PQ distortion (opq      ) = {traj[-1]:.4f}")
opq.test_opq_beats_balanced_heuristic()

## Act II — the loss

### 5. Score-aware quantization preserves the *parallel* residual

For maximum-inner-product search, the error that matters is in the inner product seen by *aligned*
(high-scoring) queries, which is dominated by the residual's component **parallel** to the
datapoint. Of two reconstructions with equal Euclidean error, score-aware (anisotropic) loss prefers
the one with the smaller parallel residual — and aligned-query inner-product error confirms it.

In [ ]:
opq.test_anisotropic_penalizes_parallel()

### 6. A score-aware codebook on the finance cloud

We train two codebooks of the same size on the unit-normalized finance cloud — isotropic $k$-means
and the anisotropic quantizer `anisotropic_lloyd` — and compare the inner-product error on each
query's true top-$k$ pairs, plus MIPS recall. The score-aware codebook optimizes the loss that
matters, so high-score inner products survive quantization better. Direction, not a magic number.

In [ ]:
d = opq.score_aware_demo()
print(f"  isotropic   k-means : high-score IP-MSE = {d['iso_ip_mse']:.5f}, recall@{d['topk']} = {d['iso_recall']:.3f}")
print(f"  anisotropic (eta={d['eta']:g}): high-score IP-MSE = {d['aniso_ip_mse']:.5f}, recall@{d['topk']} = {d['aniso_recall']:.3f}")
opq.test_anisotropic_encoding_lowers_highscore_error()

### 7. Viz constants

The exact numbers `OptimizedProductQuantizationLaboratory.tsx` mirrors to the decimal — the rotation
quartet and per-subspace variance bars (Panel A), the convergence trajectory (Panel B), and the
score-aware decomposition and $\eta$ sweep (Panel C).

In [ ]:
opq.viz_constants()

## All claims verified

Every theorem and honest caveat is an executed assertion above. The rotation distortions, the
monotone trajectory, the score-aware decomposition, and the finance comparison are the exact numbers
the laboratory mirrors. Re-run top to bottom to reproduce the topic.